# Задача трёх тел

Положение системы описывается 12-мерным вектором $\mathbf x = \left( \mathbf r_1, \mathbf r_2, \mathbf r_3, \mathbf v_1, \mathbf v_2, \mathbf v_3 \right)^T$.

Система для задачи трёх тел подчиняется следующей системе дифференциальных уравнений:

\begin{equation*}
\begin{cases}
m_1 \ddot{\mathbf r}_1 = \mathbf F_{21} + \mathbf F_{31}\\
m_2 \ddot{\mathbf r}_2 = \mathbf F_{12} + \mathbf F_{32}\\
m_3 \ddot{\mathbf r}_3 = \mathbf F_{13} + \mathbf F_{23}
\end{cases}
\end{equation*}

где $$\mathbf{F}_{ij} = \gamma \frac{m_1 m_2 \left( \mathbf r_1 - \mathbf r_2 \right)}{\left \Vert \mathbf r_1 - \mathbf r_2 \right \Vert^3}.$$

Или

\begin{equation*}
\begin{cases}
\dot{\mathbf r}_1 = \mathbf{v}_1\\
\dot{\mathbf r}_2 = \mathbf{v}_2\\
\dot{\mathbf r}_3 = \mathbf{v}_3\\
\dot{\mathbf v}_1 = \frac{1}{m_1}\left(\mathbf F_{21} + \mathbf F_{31}\right)\\
\dot{\mathbf v}_2 = \frac{1}{m_2}\left(\mathbf F_{12} + \mathbf F_{32}\right)\\
\dot{\mathbf v}_3 = \frac{1}{m_3}\left(\mathbf F_{13} + \mathbf F_{23}\right)
\end{cases}
\end{equation*}

$$\dot {\mathbf x} = \mathbf f(\mathbf x)$$

\begin{equation*}
\mathbf f(\mathbf x) =
\begin{bmatrix}
\mathbf{v}_1\\
\mathbf{v}_2\\
\mathbf{v}_3\\
\frac{1}{m_1}\left(\mathbf F_{21} + \mathbf F_{31}\right)\\
\frac{1}{m_2}\left(\mathbf F_{12} + \mathbf F_{32}\right)\\
\frac{1}{m_3}\left(\mathbf F_{13} + \mathbf F_{23}\right)
\end{bmatrix}
\end{equation*}

Импортируем необходимые модули

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

Задание системы, начальных условий и численный метод (реализуем метод Рунге-Кутты 4 порядка):

In [ ]:
def gravitational_force(r1, r2, m1, m2, gamma):
    return gamma * m1 * m2 / np.linalg.norm(r1 - r2) ** 3 * (r1 - r2)

def f(x, m1, m2, m3, gamma):
    r1 = x[:2]
    r2 = x[2:4]
    r3 = x[4:6]
    v1 = x[6:8]
    v2 = x[8:10]
    v3 = x[10:12]
    dot_v1 = 1.0 / m1 * (gravitational_force(r2, r1, m2, m1, gamma) + gravitational_force(r3, r1, m3, m1, gamma))
    dot_v2 = 1.0 / m2 * (gravitational_force(r1, r2, m1, m2, gamma) + gravitational_force(r3, r2, m3, m2, gamma))
    dot_v3 = 1.0 / m3 * (gravitational_force(r1, r3, m1, m3, gamma) + gravitational_force(r2, r3, m2, m3, gamma))
    return np.concatenate([v1, v2, v3, dot_v1, dot_v2, dot_v3])

def simulate(m1, m2, m3, x0, n_steps, T, gamma=1):
    dt = T / n_steps

    x = np.zeros((12, n_steps + 1))
    x[:, 0] = x0

    for i in range(n_steps):
        k1 = f(x[:, i], m1, m2, m3, gamma)
        k2 = f(x[:, i] + dt / 2 * k1, m1, m2, m3, gamma)
        k3 = f(x[:, i] + dt / 2 * k2, m1, m2, m3, gamma)
        k4 = f(x[:, i] + dt * k3, m1, m2, m3, gamma)
        x[:, i + 1] = x[:, i] + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    
    return x

Зададим начальные условия для устойчивой периодической конфигурации трёх тел:

In [ ]:
T = 30.0
n_steps = 300
m1 = 0.10
m2 = 0.20
m3 = 1.0
x0 = np.array([-2.59038883768724, 0,
               1.0, 0.0,
               0.0, 0.0,
               0, -0.619538016547757,
               0, -0.865730420457027,
               0, 0.235099885746181])

x = simulate(m1, m2, m3, x0, n_steps, T)

### Графики модулей скорости

In [ ]:
fig = plt.figure()

times = np.linspace(0, T, n_steps + 1)
v1_norm, v2_norm, v3_norm = [], [], []
for i in range(n_steps+1):
    v1_norm.append(np.linalg.norm(x[6:8, i]))
    v2_norm.append(np.linalg.norm(x[8:10, i]))
    v3_norm.append(np.linalg.norm(x[10:12, i]))

plt.plot(times, v1_norm, c='green')
plt.plot(times, v2_norm, c='red')
plt.plot(times, v3_norm, c='blue')

plt.show()

### Визуализация траектории системы

In [ ]:
def visualize(x, n_steps, xlim=3, ylim=3, save_gif=False, gif_name='three_body_problem.gif'):
    fig, ax = plt.subplots()
    ax.set_xlim(-xlim, xlim)
    ax.set_ylim(-ylim, ylim)
    ax.set_aspect('equal')

    line1, = ax.plot([], [], 'o-', c='green', alpha=1.0, markevery=[-1])
    line2, = ax.plot([], [], 'o-', c='red', alpha=1.0, markevery=[-1])
    line3, = ax.plot([], [], 'o-', c='blue', alpha=1.0, markevery=[-1])

    def animate(i, x):
        line1.set_data(x[0, :i+1], x[1, :i+1])
        line2.set_data(x[2, :i+1], x[3, :i+1])
        line3.set_data(x[4, :i+1], x[5, :i+1])
        return line1, line2, line3,

    ani = animation.FuncAnimation(fig, animate, frames=n_steps+1, interval=50, fargs=(x,))

    if save_gif:
        ani.save(gif_name)

    return ani

    

In [ ]:
ani = visualize(x, n_steps)
HTML(ani.to_jshtml())

In [ ]:
m1 = 0.44
m2 = 0.87
m3 = 1.0
T = 30.0
n_steps = 300
r1 = np.array([-1.21992948117021, 0.0])
r2 = np.array([ 1.0, 0.0])
r3 = np.array([ 0.0, 0.0])
v1 = np.array([0.0, -0.992252134619392])
v2 = np.array([0.0, -0.513024298255905])
v3 = np.array([0.0,  0.882922078715170])
x0 = np.hstack([r1, r2, r3, v1, v2, v3])

x = simulate(m1, m2, m3, x0, n_steps, T)

ani = visualize(x, n_steps)
HTML(ani.to_jshtml())

### Другие конфигурации

#### Одно тело улетает от двух других

In [ ]:
m1 = 1.0
m2 = 1.0
m3 = 0.2
T = 30.0
n_steps = 300
r1 = np.array([-1.0, 0.0])
r2 = np.array([1.0, 0.0])
r3 = np.array([0.1, 0.0])
v1 = np.array([0.0, 0.5])
v2 = np.array([0.0, -0.5])
v3 = np.array([0.0,  0.0])
x0 = np.hstack([r1, r2, r3, v1, v2, v3])

x = simulate(m1, m2, m3, x0, n_steps, T)

ani = visualize(x, n_steps)
HTML(ani.to_jshtml())

#### Тела демонстрируют непредсказуемую динамику

In [ ]:
r1 = np.array([-1.0, 0.367])
r2 = np.array([0.284, 0.587])
r3 = np.array([0.724, -0.954])
v1 = np.array([0.0, 0.0])
v2 = np.array([0.0, 0.0])
v3 = np.array([0.0, 0.0])
m1 = 1.0
m2 = 1.0
m3 = 1.0
T = 15.0
n_steps = 150000
x0 = np.hstack([r1, r2, r3, v1, v2, v3])

x = simulate(m1, m2, m3, x0, n_steps, T)

ani = visualize(x[:, ::1000], n_steps // 1000)
HTML(ani.to_jshtml())